# Phase 4: Nucleotide Transformer fine-tuning
Run with a Colab GPU. The Phase 2 dataset must be uploaded to `MyDrive/variantfx/data/labeled_split_dataset.tsv`; checkpoints, reports, and MLflow runs persist under `MyDrive/variantfx/phase4`. The notebook contains orchestration only; model logic remains in `src/models/finetune_lm.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
!pip install -q transformers==4.42.3 tokenizers==0.19.1 huggingface-hub==0.23.4 mlflow==2.14.1 pandas==2.2.2 scikit-learn==1.5.0
!git clone https://github.com/djasleen15/genomic-variant-prioritizer.git /content/genomic-variant-prioritizer || git -C /content/genomic-variant-prioritizer pull
%cd /content/genomic-variant-prioritizer

In [ ]:
from pathlib import Path
dataset = Path('/content/drive/MyDrive/variantfx/data/labeled_split_dataset.tsv')
output = Path('/content/drive/MyDrive/variantfx/phase4/artifacts')
mlruns = Path('/content/drive/MyDrive/variantfx/phase4/mlruns')
assert dataset.exists(), f'Upload the Phase 2 dataset to {dataset}'
output.mkdir(parents=True, exist_ok=True)
mlruns.mkdir(parents=True, exist_ok=True)

Start with full fine-tuning. If it runs out of GPU memory or cannot complete within the Colab session, rerun with `--freeze-encoder` as the documented compute fallback. Reduce `--batch-size` before freezing if the failure is memory-only.

In [ ]:
!python -m src.models.finetune_lm --input "{dataset}" --output-dir "{output}" --mlflow-dir "{mlruns}" --batch-size 16 --epochs 3 --patience 1 --audit-samples 100

In [ ]:
import json
report = json.loads((output / 'phase4_report.json').read_text())
print(json.dumps({k: report[k] for k in ['training_approach', 'validation', 'test', 'baseline', 'improvement', 'mlflow_run_id']}, indent=2))